# 분석환경 준비

데이콘에서 제공하는 '[화학] 와인 품질 분류'데이터셋을 활용한다.

In [2]:
## 필요한 기본 라이브러리를 불러오고 랜덤 시드를 고정한다.

# 필수 라이브러리
import pandas as pd
import numpy as np
import random
import tensorflow as tf

# 랜덤 시드 고정
SEED=12
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)  
print("시드 고정: ", SEED)

시드 고정:  12


# 데이터 전처리

In [4]:
#wine 폴더에 업로드한 3개의 CSV파일을 데이터 프레임으로 변환한다.


train = pd.read_csv(r"F:\pythone_deeplearning\wines\train.csv")
test = pd.read_csv(r"F:\pythone_deeplearning\wines\test.csv")
submission = pd.read_csv(r"F:\pythone_deeplearning\wines\sample_submission.csv")

print(train.shape, test.shape, submission.shape)

(5497, 14) (1000, 13) (1000, 2)


In [5]:
## train 데이터의 내용을 살펴본다. 목표 변수는 와인 품질을 나타내는 quality열이다.
train.head(2)

,index,quality,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,0,5,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,white
1,1,5,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,red


In [ ]:
## 제출 파일의 양식을 보면 와인 품질을 나타내는 quality 열에 예측값을 입력해야 한다.

submission.head()

,index,quality
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


In [7]:
## type 열의 데이터를 살펴본다. 화이트 와인이 4159개, 레드 와인이 1338개이다.

train['type'].value_counts()

type
white    4159
red      1338
Name: count, dtype: int64

In [8]:
## type 열의 범주형 데이터는 문자열 값을 갖는다.
## 모델 학습에 입력하려면 숫자형 데이터로 변환해야 한다.
## 화이트 와인을 나타내는 'white'문자열을 숫자 1로 바꾸고, 레드 와인을 나타내는 'red'문자열을 숫자 0으로 변환한다.
 

train['type'] = np.where(train['type']=='white', 1, 0).astype(int)
test['type'] = np.where(test['type']=='white', 1, 0).astype(int)
train['type'].value_counts()

type
1    4159
0    1338
Name: count, dtype: int64

In [9]:
## 이번에는 목표 변수인 quality 열의 데이터 개수를 확인한다. 6등급 와인의 개수가 가장 많다.

train['quality'].value_counts()

quality
6    2416
5    1788
7     924
4     186
8     152
3      26
9       5
Name: count, dtype: int64

목표 변수는 연속형 숫자 데이터가 아니라. 와인 등급을 나타내는 범주형 데이터이다.
케라스 to_categorical 함수를 이용하여 목표 변수를 원핫 인코딩 변환한다.

In [ ]:
## 원핫 인코딩을 하기 전에 숫자 3을 차감하여 와인 등급을 0~6 범위로 바꾼다.
## 와인 등급은 3~9까지 모두 7개 클래스로 구분되는데, 3~9 범위 값으로 원핫 인코딩을 하면
## 숫자 0부터 최대값인 9까지 10개 클래스로 인식하기 때문이다.

from tensorflow.keras.utils import to_categorical

y_train = to_categorical(train.loc[:, 'quality'] - 3)
y_train

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(5497, 7))

In [ ]:
# 피처 선택
X_train = train.loc[:, 'fixed acidity':]
X_test = test.loc[:, 'fixed acidity':]

# 피처 스케일링
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler()
scaler.fit(X_train)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

print(X_train_scaled.shape, y_train.shape)
print(X_test_scaled.shape)

# 신경망 학습

In [ ]:
# 심층 신경망 모델
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

def build_model(train_data, train_target):
    model = Sequential()
    model.add(Dense(128, activation='tanh', input_dim=train_data.shape[1]))
    model.add(Dropout(0.2))
    model.add(Dense(64, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='tanh'))
    model.add(Dense(train_target.shape[1], activation='softmax'))

    model.compile(optimizer='RMSprop', loss='categorical_crossentropy', 
                metrics=['acc', 'mae'])

    return model

model = build_model(X_train_scaled, y_train)
model.summary()

# 콜백 함수

In [ ]:
# Early Stopping 기법
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

X_tr, X_val, y_tr, y_val = train_test_split(X_train_scaled, y_train, test_size=0.15, 
                                            shuffle=True, random_state=SEED)

early_stopping = EarlyStopping(monitor='val_loss',  patience=10)
history = model.fit(X_tr, y_tr, batch_size=64, epochs=200,
                    validation_data=(X_val, y_val),
                    callbacks=[early_stopping],                    
                    verbose=2)

In [ ]:
model.evaluate(X_val, y_val)

In [ ]:
# test 데이터에 대한 예측값 정리
y_pred_proba = model.predict(X_test_scaled)
y_pred_proba[:5]

In [ ]:
y_pred_label = np.argmax(y_pred_proba, axis=-1) + 3
y_pred_label[:5]

In [ ]:
# 제출양식에 맞게 정리
submission['quality'] = y_pred_label.astype(int)
submission.head()

In [ ]:
# 제출파일 저장    
submission.to_csv(drive_path + "wine/wine_dnn_001.csv", index=False)   